# 01 — BDD100K Dataset Setup & Download

**Goal:** Identify the legitimate BDD100K source, explain what to download, and verify files.

## BDD100K Overview

| Item | Detail |
|---|---|
| **Full name** | Berkeley DeepDrive 100K |
| **Official site** | [bdd-data.berkeley.edu](https://bdd-data.berkeley.edu) |
| **Images** | 100,000 driving images (1280×720) |
| **Split** | 70K train / 10K val / 20K test |
| **Detection classes** | 10: person, rider, car, truck, bus, train, motorcycle, bicycle, traffic light, traffic sign |
| **License** | BSD 3-Clause (research use) |

## What to Download

### Required NOW (object detection)
| File | Description | Size |
|---|---|---|
| `bdd100k_images_100k.zip` | All 100K images | ~6.4 GB |
| `bdd100k_labels_release.zip` | Detection labels (JSON) | ~100 MB |

### Useful LATER (lane marking — do NOT train on yet)
| File | Description |
|---|---|
| `bdd100k_lane_marks_*.json` | Lane polyline annotations |
| `bdd100k_drivable_maps.zip` | Drivable area segmentation masks |

> 💡 Download the lane/drivable files now if bandwidth allows — they'll be used in future notebooks.

## 1 — Install Dependencies

In [ ]:
!pip install -q gdown tqdm pyyaml

## 2 — Mount Google Drive

BDD100K is large (~6.5 GB images). **Store it on Google Drive** so you don't re-download each Colab session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3 — Path Configuration

In [ ]:
import os

# ── Configure these paths ──────────────────────────────────────────
# Root directory on Google Drive for all EcoCAR data
ECOCAR_ROOT = "/content/drive/MyDrive/EcoCAR"

# Where BDD100K raw data will be stored
BDD_RAW_DIR = os.path.join(ECOCAR_ROOT, "datasets", "bdd100k_raw")

# Subdirectories
BDD_IMAGES_DIR = os.path.join(BDD_RAW_DIR, "images", "100k")
BDD_LABELS_DIR = os.path.join(BDD_RAW_DIR, "labels")

# Create directories
for d in [BDD_RAW_DIR, BDD_IMAGES_DIR, BDD_LABELS_DIR]:
    os.makedirs(d, exist_ok=True)
    print(f"📁 {d}")

print(f"\n✅ Directory structure ready")

## 4 — Download Instructions

### Option A: Official website (recommended)

1. Go to **[bdd-data.berkeley.edu](https://bdd-data.berkeley.edu)**
2. Create a free account / sign in
3. Download:
   - **Images**: `bdd100k_images_100k.zip`
   - **Labels**: `bdd100k_labels_release.zip`
4. Upload the zip files to your Google Drive at the path below
5. Run the extraction cells in this notebook

### Option B: Hugging Face mirror

If the official site is unavailable, BDD100K is also hosted on Hugging Face:
- Dataset: `dgural/bdd100k` or search "BDD100K" on [huggingface.co/datasets](https://huggingface.co/datasets)

### Option C: Upload directly to Colab

If you have the files locally, upload them to Colab:
```python
from google.colab import files
uploaded = files.upload()  # Select your zip files
```

## 5 — Extract Downloaded Files

Once you've uploaded the zip files to your Drive, run the cells below.

In [ ]:
import zipfile

# ── Set paths to your downloaded zip files ──────────────────────────
# Adjust these if you placed the zips elsewhere on Drive
IMAGES_ZIP = os.path.join(ECOCAR_ROOT, "downloads", "bdd100k_images_100k.zip")
LABELS_ZIP = os.path.join(ECOCAR_ROOT, "downloads", "bdd100k_labels.zip")

def extract_if_needed(zip_path, extract_to):
    """Extract a zip file if it exists and target seems empty."""
    if not os.path.isfile(zip_path):
        print(f"⚠️ Zip not found: {zip_path}")
        print(f"   Please download and upload to this path first.")
        return False

    print(f"📦 Extracting: {os.path.basename(zip_path)}")
    print(f"   To: {extract_to}")

    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(extract_to)

    print(f"   ✅ Done!")
    return True

In [ ]:
# Extract images
extract_if_needed(IMAGES_ZIP, BDD_RAW_DIR)

In [ ]:
# Extract labels
extract_if_needed(LABELS_ZIP, BDD_RAW_DIR)

## 6 — Verify Files & Create Symlinks

After extraction, the files may not match the standard BDD100K directory layout.
Common case: images and per-image JSON labels are mixed together in `100k/train/`.

This cell:
1. Creates **symlinks** so the expected paths (`bdd100k/images/100k/train/`) resolve correctly
2. Inspects a sample JSON to confirm the **scalabel** annotation format (`name`, `frames`, `objects`)
3. Counts how many images are actually available

In [ ]:
import os

RAW = "/content/drive/MyDrive/EcoCAR/datasets/bdd100k_raw"

# ── Step 1: Create expected directory structure via symlinks ─────
# The notebooks filter by file extension, so mixed folders are fine

target_img_dir = os.path.join(RAW, "bdd100k", "images", "100k")
os.makedirs(target_img_dir, exist_ok=True)

# Symlink train → your actual folder (notebooks will filter .jpg only)
train_link = os.path.join(target_img_dir, "train")
if not os.path.exists(train_link):
    os.symlink(os.path.join(RAW, "100k", "train"), train_link)
    print(f"✅ Symlinked train images")

# Symlink val → your actual folder
val_link = os.path.join(target_img_dir, "val")
if not os.path.exists(val_link):
    os.symlink(os.path.join(RAW, "100k", "val"), val_link)
    print(f"✅ Symlinked val images")

# ── Step 2: Check the JSON label format ─────────────────────────
import json

# Look at one JSON to understand the format
train_dir = os.path.join(RAW, "100k", "train")
sample_json = [f for f in os.listdir(train_dir) if f.endswith('.json')][0]
with open(os.path.join(train_dir, sample_json), 'r') as f:
    sample = json.load(f)

print(f"\n📄 Sample JSON: {sample_json}")
print(f"   Type: {type(sample)}")
if isinstance(sample, dict):
    print(f"   Keys: {list(sample.keys())[:10]}")
elif isinstance(sample, list):
    print(f"   Length: {len(sample)}")
    if len(sample) > 0:
        print(f"   First item keys: {list(sample[0].keys()) if isinstance(sample[0], dict) else 'not a dict'}")

# ── Step 3: Count usable data ──────────────────────────────────
train_jpgs = sum(1 for f in os.listdir(train_dir) if f.endswith('.jpg'))
val_dir = os.path.join(RAW, "100k", "val")
val_jpgs = sum(1 for f in os.listdir(val_dir) if f.endswith('.jpg'))

print(f"\n📊 Usable data:")
print(f"   Train images: {train_jpgs:,}")
print(f"   Val images:   {val_jpgs:,}")
if val_jpgs == 0:
    print(f"\n⚠️  No val images. We'll split some train images for validation later.")
    print(f"   {train_jpgs:,} train images is plenty to start training!")


### 6a — Preview Raw JSON Structure

Inspect a full per-image JSON to understand the **scalabel** annotation format.

Key fields:
- `name` — image identifier (without `.jpg` extension)
- `frames[0].objects` — list of annotated objects with `category`, `box2d`, and `attributes`
- `box2d` — bounding box as `{x1, y1, x2, y2}` in pixel coordinates

In [ ]:
import json, os

RAW = "/content/drive/MyDrive/EcoCAR/datasets/bdd100k_raw"
train_dir = os.path.join(RAW, "100k", "train")

# Read a sample to understand the structure
sample_file = [f for f in os.listdir(train_dir) if f.endswith('.json')][0]
with open(os.path.join(train_dir, sample_file), 'r') as f:
    sample = json.load(f)

print(json.dumps(sample, indent=2)[:3000])


### 6b — Merge Per-Image JSONs → Combined Label Format

BDD100K labels were extracted as **individual JSON files per image** (scalabel format).
Notebook 02 expects a **single combined JSON** with all annotations.

This cell:
1. Finds all per-image JSONs that have a matching `.jpg` image
2. Converts from scalabel format (`frames[0].objects`) to combined format (`{name, labels: [{category, box2d, poly2d}]}`)
3. Counts `box2d` (detection) and `poly2d` (lane) annotations
4. Splits into **85% train / 15% val** (since val images weren't fully extracted)
5. Saves two JSON files: `bdd100k_labels_images_train.json` and `bdd100k_labels_images_val.json`

In [ ]:
import json, os, random
from tqdm import tqdm

RAW = "/content/drive/MyDrive/EcoCAR/datasets/bdd100k_raw"
train_dir = os.path.join(RAW, "100k", "train")

# Collect all per-image JSONs
json_files = sorted([f for f in os.listdir(train_dir) if f.endswith('.json')])
print(f"Found {len(json_files):,} JSON label files")

# Find which JSONs have a matching image
jpgs = set(os.path.splitext(f)[0] for f in os.listdir(train_dir) if f.endswith('.jpg'))
matched_jsons = [f for f in json_files if os.path.splitext(f)[0] in jpgs]
print(f"JSONs with matching image: {len(matched_jsons):,}")

# Convert scalabel → combined format
combined = []
has_box2d = 0
has_poly2d = 0

for jf in tqdm(matched_jsons, desc="Merging labels"):
    with open(os.path.join(train_dir, jf), 'r') as f:
        data = json.load(f)

    # Scalabel format: frames[0].objects or frames[0].labels
    frames = data.get("frames", [])
    if not frames:
        continue

    frame = frames[0]
    img_name = data.get("name", os.path.splitext(jf)[0] + ".jpg")

    # Extract labels (try both 'objects' and 'labels' keys)
    raw_labels = frame.get("objects", frame.get("labels", []))

    labels = []
    for obj in (raw_labels or []):
        label = {"category": obj.get("category", "")}

        if "box2d" in obj:
            label["box2d"] = obj["box2d"]
            has_box2d += 1

        if "poly2d" in obj:
            label["poly2d"] = obj["poly2d"]
            has_poly2d += 1

        if "attributes" in obj:
            label["attributes"] = obj["attributes"]

        labels.append(label)

    combined.append({"name": img_name, "labels": labels})

print(f"\n✅ Merged {len(combined):,} frames")
print(f"   box2d annotations:  {has_box2d:,}")
print(f"   poly2d annotations: {has_poly2d:,}")

# ── Split into train / val (85/15) ──────────────────────────────
random.seed(42)
random.shuffle(combined)
split_idx = int(len(combined) * 0.85)
train_data = combined[:split_idx]
val_data = combined[split_idx:]

print(f"\n📊 Split: {len(train_data):,} train / {len(val_data):,} val")

# Save combined JSON files
labels_dir = os.path.join(RAW, "labels")
os.makedirs(labels_dir, exist_ok=True)

train_json_path = os.path.join(labels_dir, "bdd100k_labels_images_train.json")
val_json_path = os.path.join(labels_dir, "bdd100k_labels_images_val.json")

with open(train_json_path, 'w') as f:
    json.dump(train_data, f)
print(f"💾 {train_json_path}")

with open(val_json_path, 'w') as f:
    json.dump(val_data, f)
print(f"💾 {val_json_path}")


### 6c — Create Local Symlinks & Save Paths Config

Google Drive's filesystem doesn't always support re-creating symlinks reliably.
This cell creates **local** symlinks at `/content/bdd100k_images/100k/` that point to the Drive data.

Both `train` and `val` symlinks point to the same source folder because the val split was created
from train images (images are identified by filename — the label JSONs determine the split).

A `paths_config.yaml` is also saved so notebook 02 can find the label JSONs and image directories
without guessing.

In [ ]:
import os

# Drive symlinks are flaky — use local symlinks instead
LOCAL_IMAGES = "/content/bdd100k_images/100k"
os.makedirs(LOCAL_IMAGES, exist_ok=True)

# Both train and val images live in the same Drive folder
SRC = "/content/drive/MyDrive/EcoCAR/datasets/bdd100k_raw/100k/train"

for split in ["train", "val"]:
    link = os.path.join(LOCAL_IMAGES, split)
    if not os.path.exists(link):
        os.symlink(SRC, link)

print("✅ Local symlinks created")

# ── Save paths config for all notebooks ─────────────────────────
import yaml

ECOCAR_ROOT = "/content/drive/MyDrive/EcoCAR"
RAW = os.path.join(ECOCAR_ROOT, "datasets", "bdd100k_raw")

paths = {
    "train_images_dir": LOCAL_IMAGES + "/train",
    "val_images_dir":   LOCAL_IMAGES + "/val",
    "train_labels_json": os.path.join(RAW, "labels", "bdd100k_labels_images_train.json"),
    "val_labels_json":   os.path.join(RAW, "labels", "bdd100k_labels_images_val.json"),
}

config_path = os.path.join(ECOCAR_ROOT, "paths_config.yaml")
with open(config_path, 'w') as f:
    yaml.dump(paths, f, default_flow_style=False)

print(f"💾 {config_path}")

# ── Verify ──────────────────────────────────────────────────────
print("\n" + "="*60)
for name, path in paths.items():
    if os.path.isfile(path):
        sz = os.path.getsize(path) / (1024**2)
        print(f"  ✅ {name}: {sz:.1f} MB")
    elif os.path.isdir(path):
        n_jpg = sum(1 for f in os.listdir(path) if f.endswith('.jpg'))
        print(f"  ✅ {name}: {n_jpg:,} JPGs")
    else:
        print(f"  ❌ {name}: NOT FOUND")
print("="*60)
print("\n🎯 Ready for notebook 02!")


## 7 — Inspect Detection Labels

In [ ]:
import json
import glob

# Find detection label files
label_candidates = [
    os.path.join(BDD_RAW_DIR, "bdd100k", "labels", "det_20", "det_train.json"),
    os.path.join(BDD_RAW_DIR, "bdd100k", "labels", "det_train.json"),
    os.path.join(BDD_RAW_DIR, "labels", "det_20", "det_train.json"),
    os.path.join(BDD_RAW_DIR, "labels", "det_train.json"),
    # BDD100K v1 format
    os.path.join(BDD_RAW_DIR, "bdd100k", "labels", "bdd100k_labels_images_train.json"),
    os.path.join(BDD_RAW_DIR, "labels", "bdd100k_labels_images_train.json"),
]

train_label_path = None
for candidate in label_candidates:
    if os.path.isfile(candidate):
        train_label_path = candidate
        print(f"✅ Found train labels: {candidate}")
        break

if train_label_path is None:
    print("❌ Could not find train label file. Searching...")
    json_files = glob.glob(os.path.join(BDD_RAW_DIR, "**", "*.json"), recursive=True)
    for f in json_files[:10]:
        print(f"  Found: {f}")
    print("\n  ↑ One of these should be the detection label file.")
    print("  Update the path in notebook 02 accordingly.")

In [ ]:
# Preview a few label entries
if train_label_path:
    with open(train_label_path, 'r') as f:
        data = json.load(f)

    print(f"Total frames: {len(data)}")
    print(f"\nFirst entry keys: {list(data[0].keys())}")
    print(f"\nSample entry:")
    print(json.dumps(data[0], indent=2)[:1000])

    # Count categories
    categories = set()
    for frame in data[:1000]:  # Check first 1000
        for label in (frame.get('labels') or []):
            categories.add(label.get('category', 'unknown'))

    print(f"\nCategories found (first 1000 frames): {sorted(categories)}")

### 8 — Save Configuration for Other Notebooks

In [ ]:
import yaml

# Save paths config so other notebooks can load it
config = {
    "ecocar_root": ECOCAR_ROOT,
    "bdd_raw_dir": BDD_RAW_DIR,
    "train_label_path": train_label_path,
}

config_path = os.path.join(ECOCAR_ROOT, "paths_config.yaml")
os.makedirs(os.path.dirname(config_path), exist_ok=True)
with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"✅ Path config saved: {config_path}")
print(f"\n🎯 Dataset setup complete! Proceed to notebook 02 for BDD100K preparation.")

---

### Summary

| Downloaded | For | Status |
|---|---|---|
| `bdd100k_images_100k.zip` | Detection training | ✅ Required now |
| `bdd100k_labels_release.zip` | Detection labels | ✅ Required now |
| Lane / drivable annotations | Future lane work | ⏳ Optional now |

**Next:** Run `02_bdd100k_preparation.ipynb` to convert labels to YOLO format.